In [ ]:
# ============================================================
# PARAMETERS (papermill injects these)
# ============================================================

config_path = None
run_dir = None

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import json
import yaml
import random
import numpy as np
import pandas as pd
import torch
import os

from collections import Counter

from pathlib import Path

from scripts.set_seed import set_seed

from src.model_factory import build_model

from src.dataset import MSADataset, CLASS_MAP

from src.embedder import MSAEmbedder, ESM2Embedder

from src.explainibilty import (
    make_zero_baseline,
    explain_predictions,
    print_results,
    visualize_sequence_explanations,
    visualize_attention_explanations,
    compute_attention_weights,
    compute_saliency,
    results_to_json_compatible,
    map_ecs_explanations_to_full_sequence,
    map_ecs_attention_to_full_sequence
)

In [ ]:
# ============================================================
# UTILITY: Extract ECS regions for ECS-only models
# ============================================================

def extract_ecs_regions(sequences, ecs_regions):
    """
    Extract ECS regions from sequences.
    
    Args:
        sequences: List of sequence strings
        ecs_regions: List of [start, end] tuples (1-indexed, inclusive)
    
    Returns:
        List of concatenated ECS region substrings
    """
    if not ecs_regions:
        return sequences  # No ECS extraction needed
    
    ecs_seqs = []
    for seq in sequences:
        # Convert 1-indexed regions to 0-indexed slices
        ecs_parts = [seq[start-1:end] for start, end in ecs_regions]
        ecs_seq = "".join(ecs_parts)
        ecs_seqs.append(ecs_seq)
    return ecs_seqs

# Check if we're in ECS-only mode
is_ecs_only = cfg['data']['dataset_type'] == 'ecs_only'

# Define ECS regions for each test set
# test_data1: always use hardcoded regions (fixed reference dataset)
ecs_regions_test1 = [[28, 81], [139, 164]] if is_ecs_only else []

# test_data2: use regions from config
ecs_regions_test2 = cfg['data'].get('ecs_only_regions', []) if is_ecs_only else []

# Store original sequence length for full-sequence visualization
original_seq_len_1 = None  # Will be set after loading test_data1
original_seq_len_2 = None  # Will be set after loading test_data2

if is_ecs_only:
    print(f"ECS-only mode enabled.")
    print(f"  test_data1 (fixed reference): {ecs_regions_test1}")
    print(f"  test_data2 (from config): {ecs_regions_test2}")
else:
    print("Full-sequence mode (or no ECS regions specified)")

In [ ]:
# ============================================================
# LOAD AND EMBED TEST DATA
# ============================================================

# Inference on a fixed set of sequences
test_data1 = MSADataset(['/content/drive/MyDrive/Thesis data/MSAs/ex_claudin_msa.fasta'],
    [-1], test_data=True,
)
test_data1_seq_len = test_data1.getSequenceLength()
test_data1_seq_ids, test_data1_seqs = test_data1.getSequences()

# Store original sequence length before ECS extraction
original_seq_len_1 = len(test_data1_seqs[0]) if test_data1_seqs else test_data1_seq_len
original_seqs_1 = test_data1_seqs.copy()  # Keep original for display later

# Extract ECS regions if in ECS-only mode (using hardcoded regions for test_data1)
if is_ecs_only and ecs_regions_test1:
    test_data1_seqs = extract_ecs_regions(test_data1_seqs, ecs_regions_test1)
    test_data1_seq_len = len(test_data1_seqs[0]) if test_data1_seqs else test_data1_seq_len

print(f'# Test sequences (test_data1): {len(test_data1_seqs)}')
print(f'Sequence length (after ECS extraction if applicable): {test_data1_seq_len}')
if is_ecs_only and ecs_regions_test1:
    print(f'Original full sequence length: {original_seq_len_1}')

# Instantiate embedder based on top-level config
embedder_cfg = cfg.get('embedder', {}) or {}
embedder_name = (embedder_cfg.get('name') or 'msa_transformer').lower()

if embedder_name == 'esm2':
    embedder = ESM2Embedder(device=device)
    use_msa = False
else:
    embedder = MSAEmbedder(device=device)
    use_msa = embedder_cfg.get('use_msa_mode', cfg['data'].get('use_msa_mode', True))

if use_msa:
    test_data1_embeddings = embedder.embed_msa(sequences=test_data1_seqs, seq_length=test_data1_seq_len, max_msa_depth=len(test_data1_seqs))
else:
    test_data1_embeddings = embedder.embed_sequences_per_residue(sequences=test_data1_seqs, seq_length=test_data1_seq_len, batch_size=1)

print(f'Embeddings shape: {test_data1_embeddings.shape}')

# Run inference with final model
model.eval()
with torch.no_grad():
    logits1 = model(test_data1_embeddings.to(device))
    probs1 = torch.softmax(logits1, dim=1)
    pred1 = probs1.argmax(dim=1)

for i, cls in enumerate(pred1.cpu().numpy()):
    print(f"\n({i}) {test_data1_seq_ids[i]}:")
    if is_ecs_only and ecs_regions_test1:
        print(f"    Original sequence: {original_seqs_1[i]}")
        print(f"    ECS-only sequence: {test_data1_seqs[i]}")
    else:
        print(f"    Sequence: {test_data1_seqs[i]}")
    print(f"    Predicted class: {CLASS_MAP[cls]}, confidence={probs1[i, cls]:.3f}")

# Save preds 
if is_ecs_only and ecs_regions_test1:
    preds_df = pd.DataFrame({
        "seq_id": test_data1_seq_ids,
        "sequence": original_seqs_1 if is_ecs_only and ecs_regions_test1 else test_data1_seqs,
        "ecs_only_region": test_data1_seqs,
        "predicted_class": [CLASS_MAP[cls] for cls in pred1.cpu().numpy()],
        "confidence": probs1.cpu().numpy().tolist(),  # confidence across all classes
    })
else:
    preds_df = pd.DataFrame({
        "seq_id": test_data1_seq_ids,
        "sequence": original_seqs_1 if is_ecs_only and ecs_regions_test1 else test_data1_seqs,
        "predicted_class": [CLASS_MAP[cls] for cls in pred1.cpu().numpy()],
        "confidence": probs1.cpu().numpy().tolist(),  # confidence across all classes
    })
os.makedirs(RUN_DIR / "inference/predictions", exist_ok=True)
preds_df.to_csv(RUN_DIR / "inference/predictions/test_predictions1.csv", index=False)

In [ ]:
# ============================================================
# EXPLAINIBILTY - IG
# ============================================================

test_data1_seq_ids_to_explain = [test_data1_seq_ids[i] for i in [6, 15, 17]]
test_data1_seqs_to_explain = [test_data1_seqs[i] for i in [6, 15, 17]]
test_data1_seqs_to_explain_original = [original_seqs_1[i] for i in [6, 15, 17]] if is_ecs_only and ecs_regions_test1 else test_data1_seqs_to_explain
test_data1_embeddings_to_explain = test_data1_embeddings[[6, 15, 17], :, :]

baseline_embedding1 = make_zero_baseline(test_data1_embeddings_to_explain.shape[1], embed_dim=embedder.embedding_dim)

predicted_classes1 = pred1[[6, 15, 17]]
confidences1 = probs1[[6, 15, 17]].max(dim=1)[0]

true_classes1 = predicted_classes1  # Assume model is correct for IG

# ── Compute IG explanations ──
results1 = explain_predictions(
    model,
    test_data1_seq_ids_to_explain,
    test_data1_seqs_to_explain,
    test_data1_embeddings_to_explain,
    baseline_embedding1,
    predicted_classes1,
    confidences1,
    true_classes1,
    k=10, n_steps=100, device=device, run_ablation=True,
)

# Map ECS-only attributions back to full sequence for visualization (using hardcoded regions for test_data1)
if is_ecs_only and ecs_regions_test1:
    results1 = map_ecs_explanations_to_full_sequence(
        results1, ecs_regions_test1, original_seq_len_1
    )
    for sample in results1["samples"]:
        sample_idx = sample["sample_id"]
        sample["sequence"] = test_data1_seqs_to_explain_original[sample_idx]
    
    # Also update the display sequences for visualizations
    test_data1_seqs_to_explain = test_data1_seqs_to_explain_original

print_results(results1)

# Save explanations - convert to JSON-compatible format here
os.makedirs(os.path.dirname(RUN_DIR / "inference/explanations/explanations1_ig.json"), exist_ok=True)
with open(RUN_DIR / "inference/explanations/explanations1_ig.json", "w") as f:
    json.dump(results_to_json_compatible(results1), f, indent=2)

In [ ]:
visualize_sequence_explanations(results=results1, class_names=list(CLASS_MAP.values()), max_sequences=5, save_name=RUN_DIR / "inference/explanations/explanations1_ig_viz")

In [ ]:
# ============================================================
# EXPLAINIBILTY - ATTENTION / SALIENCY
# ============================================================

if cfg['model']['uses_attention']:
    _, attention_weights1 = compute_attention_weights(model, test_data1_embeddings_to_explain.to(device))
    save_name = RUN_DIR / "inference/explanations/explanations1_attn"
    is_saliency = False
else:
    _, attention_weights1 = compute_saliency(model, test_data1_embeddings_to_explain.to(device))
    save_name = RUN_DIR / "inference/explanations/explanations1_saliency"
    is_saliency = True

# Map ECS-only attention/saliency scores back to full sequence (using hardcoded regions for test_data1)
if is_ecs_only and ecs_regions_test1:
    attention_weights1 = map_ecs_attention_to_full_sequence(attention_weights1, ecs_regions_test1, original_seq_len_1)

visualize_attention_explanations(attention_weights1, test_data1_seqs_to_explain, test_data1_seq_ids_to_explain, predicted_classes1, confidences1, true_classes1, class_names=list(CLASS_MAP.values()), save_name=save_name, is_saliency=is_saliency)

In [ ]:
# ============================================================
# LOAD AND EMBED TEST DATA SPECIFIED IN CONFIG
# ============================================================

# Inference on a fixed set of sequences
test_data2 = MSADataset([f'/content/drive/MyDrive/Thesis data/MSAs/{cfg["evaluation"]["test_data"]}'],
    [-1], test_data=True,
 )
test_data2_seq_len = test_data2.getSequenceLength()
test_data2_seq_ids, test_data2_seqs = test_data2.getSequences()

# Store original sequence length before ECS extraction
original_seq_len_2 = len(test_data2_seqs[0]) if test_data2_seqs else test_data2_seq_len
original_seqs_2 = test_data2_seqs.copy()  # Keep original for display later

# Extract ECS regions if in ECS-only mode (using config-specified regions for test_data2)
if is_ecs_only and ecs_regions_test2:
    test_data2_seqs = extract_ecs_regions(test_data2_seqs, ecs_regions_test2)
    test_data2_seq_len = len(test_data2_seqs[0]) if test_data2_seqs else test_data2_seq_len

print(f'# Test sequences (test_data2): {len(test_data2_seqs)}')
print(f'Sequence length (after ECS extraction if applicable): {test_data2_seq_len}')
if is_ecs_only and ecs_regions_test2:
    print(f'Original full sequence length: {original_seq_len_2}')

# Embed either in MSA mode or independently depending on the config
if use_msa:
    test_data2_embeddings = embedder.embed_msa(sequences=test_data2_seqs, seq_length=test_data2_seq_len, max_msa_depth=len(test_data2_seqs))
else:
    test_data2_embeddings = embedder.embed_sequences_per_residue(sequences=test_data2_seqs, seq_length=test_data2_seq_len, batch_size=1)

print(f'Embeddings shape: {test_data2_embeddings.shape}')

# Run inference with final model
model.eval()
with torch.no_grad():
    logits2 = model(test_data2_embeddings.to(device))
    probs2 = torch.softmax(logits2, dim=1)
    pred2 = probs2.argmax(dim=1)

for i, cls in enumerate(pred2.cpu().numpy()):
    print(f"\n({i}) {test_data2_seq_ids[i]}:")
    if is_ecs_only and ecs_regions_test2:
        print(f"    Original sequence: {original_seqs_2[i]}")
        print(f"    ECS-only sequence: {test_data2_seqs[i]}")
    else:
        print(f"    Sequence: {test_data2_seqs[i]}")
    print(f"    Predicted class: {CLASS_MAP[cls]}, confidence={probs2[i, cls]:.3f}")

# Save preds 
if is_ecs_only and ecs_regions_test2:
    preds_df = pd.DataFrame({
        "seq_id": test_data2_seq_ids,
        "sequence": original_seqs_2 if is_ecs_only and ecs_regions_test2 else test_data2_seqs,
        "ecs_only_region": test_data2_seqs,
        "predicted_class": [CLASS_MAP[cls] for cls in pred2.cpu().numpy()],
        "confidence": probs2.cpu().numpy().tolist(),  # confidence across all classes
    })
else:
    preds_df = pd.DataFrame({
        "seq_id": test_data2_seq_ids,
        "sequence": original_seqs_2 if is_ecs_only and ecs_regions_test2 else test_data2_seqs,
        "predicted_class": [CLASS_MAP[cls] for cls in pred2.cpu().numpy()],
        "confidence": probs2.cpu().numpy().tolist(),  # confidence across all classes
    })
preds_df.to_csv(RUN_DIR / "inference/predictions/test_predictions2.csv", index=False)

In [ ]:
# ============================================================
# EXPLAINIBILTY - IG
# ============================================================

baseline_embedding2 = make_zero_baseline(test_data2_embeddings.shape[1], embed_dim=embedder.embedding_dim)

confidences2 = probs2.max(1)[0]

true_classes2 = pred2  # Assume model is correct for IG
test_data2_seqs_original = original_seqs_2.copy() if is_ecs_only and ecs_regions_test2 else test_data2_seqs

# ── Compute IG explanations ──
results2 = explain_predictions(
    model,
    test_data2_seq_ids,
    test_data2_seqs,
    test_data2_embeddings,
    baseline_embedding2,
    pred2,
    confidences2,
    true_classes2,
    k=10, n_steps=100, device=device, run_ablation=True,
)

# Map ECS-only attributions back to full sequence for visualization (using config-based regions for test_data2)
if is_ecs_only and ecs_regions_test2:
    results2 = map_ecs_explanations_to_full_sequence(
        results2, ecs_regions_test2, original_seq_len_2
    )
    for sample in results2["samples"]:
        sample_idx = sample["sample_id"]
        sample["sequence"] = test_data2_seqs_original[sample_idx]
    
    # Also update the display sequences for visualizations
    test_data2_seqs = test_data2_seqs_original

print_results(results2)

# Save explanations - convert to JSON-compatible format here
os.makedirs(os.path.dirname(RUN_DIR / "inference/explanations/explanations2_ig.json"), exist_ok=True)
with open(RUN_DIR / "inference/explanations/explanations2_ig.json", "w") as f:
    json.dump(results_to_json_compatible(results2), f, indent=2)

In [ ]:
visualize_sequence_explanations(results=results2, class_names=list(CLASS_MAP.values()), max_sequences=5, save_name=RUN_DIR / "inference/explanations/explanations2_ig_viz")

In [ ]:
# ============================================================
# EXPLAINIBILTY - ATTENTION / SALIENCY (TEST DATA 2)
# ============================================================

if cfg['model']['uses_attention']:
    _, attention_weights2 = compute_attention_weights(model, test_data2_embeddings.to(device))
    save_name = RUN_DIR / "inference/explanations/explanations2_attn"
    is_saliency = False
else:
    _, attention_weights2 = compute_saliency(model, test_data2_embeddings.to(device))
    save_name = RUN_DIR / "inference/explanations/explanations2_saliency"
    is_saliency = True

# Map ECS-only attention/saliency scores back to full sequence (using config-based regions for test_data2)
if is_ecs_only and ecs_regions_test2:
    attention_weights2 = map_ecs_attention_to_full_sequence(attention_weights2, ecs_regions_test2, original_seq_len_2)

visualize_attention_explanations(attention_weights2, test_data2_seqs, test_data2_seq_ids, pred2, confidences2, true_classes2, class_names=list(CLASS_MAP.values()), save_name=save_name, is_saliency=is_saliency)